In [1]:
import numpy as np
import h5py
from scipy.ndimage import zoom

In [2]:
raw_path = {
    "source": "2D_DarcyFlow_beta0.01_Train.hdf5", #huggingface: scaomath/navier-stokes-dataset
}
save_path = {
    "train": "trainset.h5",
    "test": "testset.h5",
}

In [3]:
def resize_batch_numpy(data, target_size=64, is_binary=False):
    """
    data: numpy array with shape [B, 1, H, W]
    target_size: int (e.g., 64)
    is_binary: True -> nearest; False -> bilinear
    """
    B, C, H, W = data.shape
    scale = target_size / H
    order = 0 if is_binary else 1  # 0=nearest, 1=bilinear

    # 插值  -> 注意 zoom 可以对所有维度一起缩放
    data_resized = zoom(data, (1, 1, scale, scale), order=order)

    return data_resized

In [4]:
def minmax_norm(x, xmin, xmax):
    """Min-Max 到 [-1, 1]"""
    return 2 * (x - xmin) / (xmax - xmin) - 1

### Trainset Process

In [5]:
with h5py.File(raw_path["source"], "r") as f:
    target = f["tensor"][:][:]
    obs_label = f["nu"][:][:]
obs_label = np.expand_dims(obs_label, axis=1)

In [6]:
valid_indices = [i for i in range(len(target)) if target[i].max() <= 0.004]

# 根据条件筛选数据
target = target[valid_indices]
obs_label = obs_label[valid_indices]

In [7]:
obs_label = resize_batch_numpy(obs_label, target_size=64, is_binary=False)
target = resize_batch_numpy(target, target_size=64, is_binary=True)
print(target.shape,obs_label.shape)

(5560, 1, 64, 64) (5560, 1, 64, 64)


In [8]:
target_trainset = target[0:5000]
target_testset = target[5000:5200]
obs_label_trainset = obs_label[0:5000]
obs_label_testset = obs_label[5000:5200]

In [9]:
target_max, target_min = target_trainset.max(), target_trainset.min()
target_trainset_norm = minmax_norm(target_trainset, target_min, target_max)
obs_max, obs_min = obs_label_trainset.max(), obs_label_trainset.min()
obs_label_train_norm = minmax_norm(obs_label_trainset, obs_min, obs_max)

### Testset Process

In [10]:
# target_max, target_min = target_trainset.max(), target_trainset.min()
target_test_norm = minmax_norm(target_testset, target_min, target_max)
# obs_max, obs_min = obs_label_trainset.max(), obs_label_trainset.min()
obs_label_testset_norm = minmax_norm(obs_label_testset, obs_min, obs_max)

In [11]:
with h5py.File(f'../{save_path["train"]}', "w") as f:
    f.create_dataset("target", data=target_trainset_norm)
    f.create_dataset("dense_obs", data=obs_label_train_norm)
    f.create_dataset("target_scaler_params", data=np.array([target_max, target_min]))
    f.create_dataset("obs_scaler_params", data=np.array([obs_max, obs_min]))
print(target_trainset_norm.shape, obs_label_train_norm.shape)

(5000, 1, 64, 64) (5000, 1, 64, 64)


In [12]:
with h5py.File(f'../{save_path["test"]}', "w") as f:
    f.create_dataset("target", data=target_test_norm)
    f.create_dataset("dense_obs", data=obs_label_testset_norm)
    f.create_dataset("target_scaler_params", data=np.array([target_max, target_min]))
    f.create_dataset("obs_scaler_params", data=np.array([obs_max, obs_min]))
    print(target_test_norm.shape, obs_label_testset_norm.shape)

(200, 1, 64, 64) (200, 1, 64, 64)
